# Encoder-Decoder Translation

This toy English-to-French example demonstrates teacher forcing: the encoder reads the source sequence and the decoder learns to predict the next target token.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

pairs = [("hello", "bonjour"), ("good morning", "bonjour"), ("thank you", "merci"), ("goodbye", "au revoir"), ("i am fine", "je vais bien")]
source_sentences = [sentence for sentence, _ in pairs]
target_sentences = ["<start> " + sentence + " <end>" for _, sentence in pairs]
source_words = sorted(set(" ".join(source_sentences).split()))
target_words = sorted(set(" ".join(target_sentences).split()))
source_ids = {word: index + 1 for index, word in enumerate(source_words)}
target_ids = {word: index + 1 for index, word in enumerate(target_words)}
source_length = max(len(sentence.split()) for sentence in source_sentences)
target_length = max(len(sentence.split()) for sentence in target_sentences)

def encode(sentences, mapping, length):
	return tf.keras.utils.pad_sequences([[mapping[word] for word in sentence.split()] for sentence in sentences], maxlen=length, padding="post")

encoder_input = encode(source_sentences, source_ids, source_length)
decoder_input = encode(target_sentences, target_ids, target_length)
decoder_target = np.roll(decoder_input, -1, axis=1)

encoder_tokens = layers.Input((source_length,))
decoder_tokens = layers.Input((target_length,))
encoder_embedding = layers.Embedding(len(source_ids) + 1, 32)(encoder_tokens)
_, state_h, state_c = layers.LSTM(64, return_state=True)(encoder_embedding)
decoder_embedding = layers.Embedding(len(target_ids) + 1, 32)(decoder_tokens)
decoder_output = layers.LSTM(64, return_sequences=True)(decoder_embedding, initial_state=[state_h, state_c])
decoder_output = layers.Dense(len(target_ids) + 1, activation="softmax")(decoder_output)
model = models.Model([encoder_tokens, decoder_tokens], decoder_output)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit([encoder_input, decoder_input], decoder_target[..., None], epochs=250, verbose=0)
print("Encoder input shape:", encoder_input.shape)
print("Decoder output shape:", model.output_shape)

prediction = model.predict([encoder_input[:1], decoder_input[:1]], verbose=0)[0].argmax(axis=-1)
inverse_target_ids = {value: word for word, value in target_ids.items()}
print("Predicted tokens:", " ".join(inverse_target_ids.get(index, "<pad>") for index in prediction if index))